# BLEnD-CultureRAG: Research Studies & Analysis

**Paper:** Trust-Weighted RAG for Cultural Multiple-Choice Question Answering (SemEval Task 7)

This notebook loads **pre-computed predictions** from the ablation study and runs all statistical analyses, visualizations, and table generation needed for the research paper. **No model inference is performed.**

### Studies Included
1. **Ablation Study** — Accuracy progression across 8 system configurations
2. **Statistical Significance** — Bootstrap CIs, McNemar's test, pairwise significance matrix
3. **Effect Size Analysis** — Cohen's h for each phase transition
4. **Country-Level Analysis** — Per-country accuracy breakdown & RAG impact
5. **RAG Impact Analysis** — Cases where RAG helped vs. hurt predictions
6. **Error Taxonomy** — Categorization of failure modes
7. **Prediction Agreement** — Cross-configuration agreement matrix
8. **LaTeX Export** — Publication-ready tables and figures

In [ ]:
# ============================================================================
# CELL 1: IMPORTS & SETUP
# ============================================================================

import pandas as pd
import numpy as np
import json
import os
import math
import warnings
from pathlib import Path
from collections import defaultdict, Counter
from scipy.stats import chi2
from itertools import combinations

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Publication-quality plot defaults ──────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.figsize': (10, 6),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_palette('colorblind')

print('All imports loaded successfully.')

In [ ]:
# ============================================================================
# CELL 2: CONFIGURATION — File Paths
# ============================================================================

# ── Root workspace directory (adjust if running from a different location) ──
ROOT = Path(r'c:\Users\LawLight\OneDrive\Desktop\semevals')

# ── Prediction data ─────────────────────────────────────────────────────────
ABLATION_DIR       = ROOT / 'ablation_predictions'
ABLATION_SUMMARY   = ABLATION_DIR / 'ablation_summary.csv'

# ── Ground truth (v5 comparison file has correct answers for all 148 IDs) ──
GROUND_TRUTH_FILE  = ROOT / 'predictions-per-version' / 'all_predictions_comparison-v5.csv'

# ── Questions & answers (full BLEnD dataset) ────────────────────────────────
QUESTIONS_FILE     = ROOT / 'blend-data' / 'questions.tsv'
ANSWERS_FILE       = ROOT / 'blend-data' / 'answers.tsv'

# ── Output directory for figures & tables ────────────────────────────────────
OUTPUT_DIR = ROOT / 'study_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / 'figures').mkdir(exist_ok=True)
(OUTPUT_DIR / 'tables').mkdir(exist_ok=True)

# ── Ordered ablation configuration names (cumulative feature addition) ──────
CONFIG_ORDER = [
    'baseline_no_rag',
    'rag_basic',
    'phase1_countryfilter',
    'phase2_intent',
    'phase3_tiered',
    'phase4_quality',
    'phase5_trust_weight',
    'phase6_full_system',
]

CONFIG_LABELS = {
    'baseline_no_rag':      'Baseline (No RAG)',
    'rag_basic':            '+ Basic RAG',
    'phase1_countryfilter': '+ Country Filter',
    'phase2_intent':        '+ Intent Detection',
    'phase3_tiered':        '+ Tiered Routing',
    'phase4_quality':       '+ Anti-Leak / Trust Prompts',
    'phase5_trust_weight':  '+ Trust-Weighted Reranking',
    'phase6_full_system':   'Full System',
}

# Short labels for figures
CONFIG_SHORT = {
    'baseline_no_rag':      'Baseline',
    'rag_basic':            '+RAG',
    'phase1_countryfilter': '+Country',
    'phase2_intent':        '+Intent',
    'phase3_tiered':        '+Tiered',
    'phase4_quality':       '+Quality',
    'phase5_trust_weight':  '+TrustWt',
    'phase6_full_system':   'Full',
}

print(f'Root:       {ROOT}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Configs:    {len(CONFIG_ORDER)}')

In [ ]:
# ============================================================================
# CELL 3: LOAD ALL PREDICTION DATA & GROUND TRUTH
# ============================================================================

# ── 1. Load ground truth from v5 comparison file ────────────────────────────
gt_df = pd.read_csv(GROUND_TRUTH_FILE)
truth_map = dict(zip(gt_df['id'], gt_df['correct']))
questions_map = dict(zip(gt_df['id'], gt_df['question']))

# Also store options
options_map = {}
for _, row in gt_df.iterrows():
    options_map[row['id']] = {
        'A': row.get('option_A', ''),
        'B': row.get('option_B', ''),
        'C': row.get('option_C', ''),
        'D': row.get('option_D', ''),
    }

print(f'Ground truth loaded: {len(truth_map)} questions')
print(f'Answer distribution: {Counter(truth_map.values())}')

# ── 2. Load all ablation predictions from CSVs ──────────────────────────────
ablation_preds = {}   # config_name -> {id: prediction}
ablation_lists = {}   # config_name -> [prediction] (ordered by sorted ID)

for config in CONFIG_ORDER:
    csv_path = ABLATION_DIR / f'{config}.csv'
    if csv_path.exists():
        df_pred = pd.read_csv(csv_path)
        pred_dict = dict(zip(df_pred['id'], df_pred['prediction']))
        ablation_preds[config] = pred_dict
        print(f'  Loaded {config}: {len(pred_dict)} predictions')
    else:
        print(f'  WARNING: {csv_path} not found')

# ── 3. Build aligned arrays (same ID order for all configs) ─────────────────
all_ids = sorted(truth_map.keys())
correct_answers = [truth_map[qid] for qid in all_ids]

for config in CONFIG_ORDER:
    if config in ablation_preds:
        ablation_lists[config] = [ablation_preds[config].get(qid, 'C') for qid in all_ids]

N = len(all_ids)
print(f'\nAligned dataset: {N} questions, {len(ablation_lists)} configs')

# ── 4. Load ablation timing summary ────────────────────────────────────────
timing_df = pd.read_csv(ABLATION_SUMMARY)
timing_map = dict(zip(timing_df['config'], timing_df['time_seconds']))
desc_map = dict(zip(timing_df['config'], timing_df['description']))
print(f'Timing data: {len(timing_map)} configs')

# ── 5. Extract country codes ────────────────────────────────────────────────
def extract_country(qid):
    if '-' in qid:
        return qid.split('-')[1].split('_')[0]
    return 'UNK'

country_list = [extract_country(qid) for qid in all_ids]
countries = sorted(set(country_list))
print(f'Countries: {len(countries)} — {countries}')

---
## Study 1: Ablation Study — Component Contribution Analysis

We evaluate the incremental contribution of each system component by progressively enabling features from a bare LLM baseline to the full CultureRAG pipeline.

In [ ]:
# ============================================================================
# CELL 4: ABLATION ACCURACY TABLE
# ============================================================================

rows = []
prev_acc = None

for config in CONFIG_ORDER:
    preds = ablation_lists[config]
    correct = sum(p == c for p, c in zip(preds, correct_answers))
    acc = correct / N * 100
    delta = acc - prev_acc if prev_acc is not None else 0.0
    
    # Wilson score 95% CI
    p_hat = correct / N
    z = 1.96
    se = math.sqrt(p_hat * (1 - p_hat) / N)
    ci_lo = max(0, (acc - z * se * 100))
    ci_hi = min(100, (acc + z * se * 100))
    
    rows.append({
        'Config': config,
        'Label': CONFIG_LABELS[config],
        'Correct': correct,
        'Total': N,
        'Accuracy (%)': round(acc, 2),
        'Delta (pp)': round(delta, 2),
        'CI Lower': round(ci_lo, 2),
        'CI Upper': round(ci_hi, 2),
        'Time (s)': round(timing_map.get(config, 0), 1),
    })
    prev_acc = acc

ablation_df = pd.DataFrame(rows)

# Total gain
baseline_acc = ablation_df.iloc[0]['Accuracy (%)']
best_acc = ablation_df['Accuracy (%)'].max()
total_gain = best_acc - baseline_acc

# Display
print('=' * 90)
print('ABLATION STUDY RESULTS')
print('=' * 90)
display_cols = ['Label', 'Correct', 'Accuracy (%)', 'Delta (pp)', 'CI Lower', 'CI Upper', 'Time (s)']
print(ablation_df[display_cols].to_string(index=False))
print(f'\nTotal accuracy gain: {baseline_acc:.1f}% → {best_acc:.1f}% (+{total_gain:.1f} pp)')

# Save
ablation_df.to_csv(OUTPUT_DIR / 'tables' / 'ablation_results.csv', index=False)
print(f'\nSaved to {OUTPUT_DIR}/tables/ablation_results.csv')

In [ ]:
# ============================================================================
# CELL 5: FIGURE 1 — Ablation Accuracy Progression (Bar Chart)
# ============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

labels = [CONFIG_SHORT[c] for c in CONFIG_ORDER]
accs = ablation_df['Accuracy (%)'].values
ci_lo = ablation_df['CI Lower'].values
ci_hi = ablation_df['CI Upper'].values
errors = np.array([accs - ci_lo, ci_hi - accs])

colors = sns.color_palette('Blues_d', len(CONFIG_ORDER))
colors[0] = sns.color_palette('Reds_d', 3)[1]  # Red for baseline
colors[-1] = sns.color_palette('Greens_d', 3)[1]  # Green for full system

bars = ax.bar(range(len(labels)), accs, color=colors, edgecolor='white', linewidth=0.8)
ax.errorbar(range(len(labels)), accs, yerr=errors, fmt='none', ecolor='black',
            capsize=4, capthick=1.2, linewidth=1.2)

# Value labels on bars
for i, (bar, acc) in enumerate(zip(bars, accs)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{acc:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Delta annotations
for i in range(1, len(accs)):
    delta = accs[i] - accs[i-1]
    if abs(delta) > 0.05:
        color = 'green' if delta > 0 else 'red'
        ax.annotate(f'{delta:+.1f}', xy=(i, accs[i] - 2),
                    fontsize=8, color=color, ha='center', style='italic')

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Figure 1: Ablation Study — Incremental Component Contribution')
ax.set_ylim(0, max(accs) + 10)
ax.axhline(y=25, color='gray', linestyle=':', alpha=0.5, label='Random baseline (25%)')
ax.legend(loc='lower right')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / 'fig1_ablation_progression.pdf', bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figures' / 'fig1_ablation_progression.png', bbox_inches='tight')
plt.show()
print('Saved: fig1_ablation_progression.pdf/png')

In [ ]:
# ============================================================================
# CELL 6: FIGURE 2 — Component Contribution Waterfall Chart
# ============================================================================

fig, ax = plt.subplots(figsize=(12, 6))

deltas = ablation_df['Delta (pp)'].values
labels = [CONFIG_SHORT[c] for c in CONFIG_ORDER]

# Waterfall: cumulative positions
cumulative = [ablation_df.iloc[0]['Accuracy (%)']]
for d in deltas[1:]:
    cumulative.append(cumulative[-1] + d)

bottom = [0] + cumulative[:-1]
bar_colors = ['#d62728']  # baseline
for d in deltas[1:]:
    bar_colors.append('#2ca02c' if d > 0 else '#d62728' if d < 0 else '#7f7f7f')

# Plot waterfall bars
heights = [cumulative[0]] + list(deltas[1:])
ax.bar(range(len(labels)), heights, bottom=[0] + list(cumulative[:-1]),
       color=bar_colors, edgecolor='white', linewidth=0.8, alpha=0.85)

# Connector lines
for i in range(len(cumulative) - 1):
    ax.plot([i + 0.4, i + 0.6], [cumulative[i], cumulative[i]],
            color='gray', linewidth=0.8, linestyle='--')

# Value labels
for i, (h, b, c) in enumerate(zip(heights, [0] + list(cumulative[:-1]), cumulative)):
    y_pos = c + 0.5
    label = f'{h:+.1f}' if i > 0 else f'{h:.1f}'
    ax.text(i, y_pos, label, ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Figure 2: Waterfall — Per-Component Accuracy Gain (pp)')
ax.axhline(y=25, color='gray', linestyle=':', alpha=0.5)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / 'fig2_waterfall_contribution.pdf', bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figures' / 'fig2_waterfall_contribution.png', bbox_inches='tight')
plt.show()
print('Saved: fig2_waterfall_contribution.pdf/png')

In [ ]:
# ============================================================================
# CELL 7: Component Contribution Pie / Horizontal Bar
# ============================================================================

# Only positive contributions
gains = ablation_df[ablation_df['Delta (pp)'] > 0][['Label', 'Delta (pp)']].copy()
gains['Pct of Total'] = gains['Delta (pp)'] / total_gain * 100
gains = gains.sort_values('Delta (pp)', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = sns.color_palette('viridis', len(gains))
ax.barh(gains['Label'], gains['Delta (pp)'], color=colors_bar, edgecolor='white')

for i, (_, row) in enumerate(gains.iterrows()):
    ax.text(row['Delta (pp)'] + 0.15, i,
            f"+{row['Delta (pp)']:.1f} pp ({row['Pct of Total']:.0f}%)",
            va='center', fontsize=9)

ax.set_xlabel('Accuracy Gain (percentage points)')
ax.set_title('Figure 3: Individual Component Contribution to Total Gain')
ax.axvline(x=0, color='black', linewidth=0.8)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / 'fig3_component_contribution.pdf', bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figures' / 'fig3_component_contribution.png', bbox_inches='tight')
plt.show()

# Print table
print('\nComponent Contribution Breakdown:')
for _, row in gains.sort_values('Delta (pp)', ascending=False).iterrows():
    print(f"  {row['Label']:35s}  +{row['Delta (pp)']:5.2f} pp  ({row['Pct of Total']:5.1f}% of total gain)")

---
## Study 2: Statistical Significance Testing

We verify that observed improvements are not due to chance using:
- **Bootstrap resampling** (10,000 iterations) for confidence intervals
- **McNemar's test** for paired significance between configurations
- **Cohen's h** for effect size magnitude

In [ ]:
# ============================================================================
# CELL 8: BOOTSTRAP CONFIDENCE INTERVALS
# ============================================================================

def bootstrap_ci(predictions, correct, n_boot=10000, confidence=0.95):
    """Bootstrap confidence interval for accuracy."""
    n = len(predictions)
    is_correct = np.array([p == c for p, c in zip(predictions, correct)])
    
    rng = np.random.default_rng(42)
    boot_accs = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        boot_accs.append(is_correct[idx].mean() * 100)
    
    boot_accs = np.array(boot_accs)
    alpha = 1 - confidence
    return {
        'mean': boot_accs.mean(),
        'std': boot_accs.std(),
        'ci_lo': np.percentile(boot_accs, alpha/2 * 100),
        'ci_hi': np.percentile(boot_accs, (1 - alpha/2) * 100),
    }

# Compute for all configs
boot_results = {}
for config in CONFIG_ORDER:
    boot_results[config] = bootstrap_ci(ablation_lists[config], correct_answers)

# ── Figure: Bootstrap CIs ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

labels = [CONFIG_SHORT[c] for c in CONFIG_ORDER]
means = [boot_results[c]['mean'] for c in CONFIG_ORDER]
ci_lo = [boot_results[c]['ci_lo'] for c in CONFIG_ORDER]
ci_hi = [boot_results[c]['ci_hi'] for c in CONFIG_ORDER]
errors = [[m - lo for m, lo in zip(means, ci_lo)],
          [hi - m for m, hi in zip(means, ci_hi)]]

ax.errorbar(range(len(labels)), means, yerr=errors,
            fmt='o', markersize=8, capsize=6, capthick=2, linewidth=2,
            color='#1f77b4', ecolor='#1f77b4', markerfacecolor='white',
            markeredgewidth=2)

# Shade baseline CI band
ax.axhspan(ci_lo[0], ci_hi[0], alpha=0.15, color='red', label='Baseline 95% CI')

for i, (m, lo, hi) in enumerate(zip(means, ci_lo, ci_hi)):
    ax.annotate(f'{m:.1f}%\n[{lo:.1f}, {hi:.1f}]',
                xy=(i, hi + 0.8), ha='center', fontsize=8)

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=30, ha='right')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Figure 4: Bootstrap 95% Confidence Intervals (10,000 resamples)')
ax.legend(loc='lower right')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / 'fig4_bootstrap_ci.pdf', bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figures' / 'fig4_bootstrap_ci.png', bbox_inches='tight')
plt.show()

# Print table
print(f"{'Config':<25} {'Mean':>8} {'95% CI':>18} {'Std':>8}")
print('-' * 62)
for c in CONFIG_ORDER:
    r = boot_results[c]
    print(f"{CONFIG_SHORT[c]:<25} {r['mean']:>7.2f}% [{r['ci_lo']:>6.2f}, {r['ci_hi']:>6.2f}] {r['std']:>7.2f}")

In [ ]:
# ============================================================================
# CELL 9: McNEMAR'S PAIRWISE SIGNIFICANCE TESTS
# ============================================================================

def mcnemar_test(preds_A, preds_B, correct):
    """McNemar's test with continuity correction."""
    b = sum(1 for a, bx, c in zip(preds_A, preds_B, correct)
            if a == c and bx != c)  # A correct, B wrong
    c_val = sum(1 for a, bx, c in zip(preds_A, preds_B, correct)
               if a != c and bx == c)  # A wrong, B correct
    
    if b + c_val == 0:
        return {'chi2': 0, 'p': 1.0, 'b': b, 'c': c_val, 'sig': False}
    
    stat = (abs(b - c_val) - 1) ** 2 / (b + c_val)
    p_val = 1 - chi2.cdf(stat, df=1)
    return {'chi2': stat, 'p': p_val, 'b': b, 'c': c_val, 'sig': p_val < 0.05}

# Build pairwise matrix
n_configs = len(CONFIG_ORDER)
p_matrix = np.ones((n_configs, n_configs))
sig_matrix = np.zeros((n_configs, n_configs))
pairwise_rows = []

for i, j in combinations(range(n_configs), 2):
    cA, cB = CONFIG_ORDER[i], CONFIG_ORDER[j]
    result = mcnemar_test(ablation_lists[cA], ablation_lists[cB], correct_answers)
    p_matrix[i, j] = result['p']
    p_matrix[j, i] = result['p']
    sig_matrix[i, j] = 1 if result['sig'] else 0
    sig_matrix[j, i] = sig_matrix[i, j]
    
    accA = sum(p == c for p, c in zip(ablation_lists[cA], correct_answers)) / N * 100
    accB = sum(p == c for p, c in zip(ablation_lists[cB], correct_answers)) / N * 100
    
    pairwise_rows.append({
        'Config A': CONFIG_SHORT[cA],
        'Config B': CONFIG_SHORT[cB],
        'Acc A (%)': round(accA, 2),
        'Acc B (%)': round(accB, 2),
        'Delta (pp)': round(accB - accA, 2),
        'p-value': result['p'],
        'Significant': '***' if result['p'] < 0.001 else '**' if result['p'] < 0.01 else '*' if result['p'] < 0.05 else 'ns',
    })

pairwise_df = pd.DataFrame(pairwise_rows)

# ── Figure: Significance Heatmap ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

short_labels = [CONFIG_SHORT[c] for c in CONFIG_ORDER]

# Log-transform p-values for better visualization
log_p = -np.log10(p_matrix + 1e-10)
np.fill_diagonal(log_p, 0)

mask = np.triu(np.ones_like(p_matrix, dtype=bool), k=0)
sns.heatmap(log_p, mask=mask, xticklabels=short_labels, yticklabels=short_labels,
            annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label': '-log10(p-value)'},
            linewidths=0.5, linecolor='white')

# Add significance stars
for i in range(n_configs):
    for j in range(i+1, n_configs):
        p = p_matrix[i, j]
        star = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
        if star:
            ax.text(j + 0.5, i + 0.8, star, ha='center', va='center',
                    fontsize=8, color='white', fontweight='bold')

ax.set_title("Figure 5: McNemar's Test — Pairwise p-values (-log10 scale)")

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / 'fig5_significance_heatmap.pdf', bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figures' / 'fig5_significance_heatmap.png', bbox_inches='tight')
plt.show()

# Print consecutive comparisons (most relevant for paper)
print('\nConsecutive Phase Comparisons:')
print(f"{'Transition':<40} {'Delta':>8} {'p-value':>10} {'Sig'}")
print('-' * 65)
for i in range(n_configs - 1):
    row = pairwise_df[(pairwise_df['Config A'] == CONFIG_SHORT[CONFIG_ORDER[i]]) &
                      (pairwise_df['Config B'] == CONFIG_SHORT[CONFIG_ORDER[i+1]])]
    if len(row) > 0:
        r = row.iloc[0]
        print(f"{r['Config A'] + ' → ' + r['Config B']:<40} {r['Delta (pp)']:>+7.2f}  {r['p-value']:>9.4f}  {r['Significant']}")

# Save
pairwise_df.to_csv(OUTPUT_DIR / 'tables' / 'pairwise_significance.csv', index=False)

In [ ]:
# ============================================================================
# CELL 10: COHEN'S h EFFECT SIZE ANALYSIS
# ============================================================================

def cohens_h(p1, p2):
    """Cohen's h effect size for two proportions."""
    p1 = max(0.001, min(0.999, p1))
    p2 = max(0.001, min(0.999, p2))
    return 2 * (math.asin(math.sqrt(p1)) - math.asin(math.sqrt(p2)))

def interpret_h(h):
    h_abs = abs(h)
    if h_abs < 0.2: return 'Small'
    elif h_abs < 0.5: return 'Medium'
    else: return 'Large'

# ── Compute effect sizes ──────────────────────────────────────────────────
effect_rows = []
p_baseline = sum(p == c for p, c in zip(ablation_lists[CONFIG_ORDER[0]], correct_answers)) / N

# vs. baseline
for config in CONFIG_ORDER[1:]:
    p_cfg = sum(p == c for p, c in zip(ablation_lists[config], correct_answers)) / N
    h = cohens_h(p_cfg, p_baseline)
    effect_rows.append({
        'Comparison': f'Baseline → {CONFIG_SHORT[config]}',
        'Type': 'vs Baseline',
        'h': round(h, 4),
        '|h|': round(abs(h), 4),
        'Interpretation': interpret_h(h),
        'Delta (pp)': round((p_cfg - p_baseline) * 100, 2),
    })

# Consecutive
for i in range(len(CONFIG_ORDER) - 1):
    cA, cB = CONFIG_ORDER[i], CONFIG_ORDER[i+1]
    pA = sum(p == c for p, c in zip(ablation_lists[cA], correct_answers)) / N
    pB = sum(p == c for p, c in zip(ablation_lists[cB], correct_answers)) / N
    h = cohens_h(pB, pA)
    effect_rows.append({
        'Comparison': f'{CONFIG_SHORT[cA]} → {CONFIG_SHORT[cB]}',
        'Type': 'Consecutive',
        'h': round(h, 4),
        '|h|': round(abs(h), 4),
        'Interpretation': interpret_h(h),
        'Delta (pp)': round((pB - pA) * 100, 2),
    })

effect_df = pd.DataFrame(effect_rows)

# ── Figure: Effect sizes ──────────────────────────────────────────────────
consec = effect_df[effect_df['Type'] == 'Consecutive'].copy()
vs_base = effect_df[effect_df['Type'] == 'vs Baseline'].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Consecutive
colors_h = ['#2ca02c' if h > 0 else '#d62728' for h in consec['h']]
ax1.barh(consec['Comparison'], consec['h'], color=colors_h, alpha=0.8)
ax1.axvline(x=0.2, color='orange', linestyle='--', alpha=0.6, label='Small (0.2)')
ax1.axvline(x=0.5, color='red', linestyle='--', alpha=0.6, label='Medium (0.5)')
ax1.axvline(x=-0.2, color='orange', linestyle='--', alpha=0.6)
ax1.set_xlabel("Cohen's h")
ax1.set_title('Consecutive Phase Transitions')
ax1.legend(fontsize=8)

# Right: vs Baseline
colors_h2 = ['#2ca02c' if h > 0 else '#d62728' for h in vs_base['h']]
ax2.barh(vs_base['Comparison'], vs_base['h'], color=colors_h2, alpha=0.8)
ax2.axvline(x=0.2, color='orange', linestyle='--', alpha=0.6, label='Small (0.2)')
ax2.axvline(x=0.5, color='red', linestyle='--', alpha=0.6, label='Medium (0.5)')
ax2.set_xlabel("Cohen's h")
ax2.set_title('Each Config vs. Baseline')
ax2.legend(fontsize=8)

fig.suptitle("Figure 6: Effect Size Analysis (Cohen's h)", fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / 'fig6_effect_size.pdf', bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figures' / 'fig6_effect_size.png', bbox_inches='tight')
plt.show()

# Print table
print('\nEffect Size Summary:')
print(effect_df[['Comparison', 'Type', 'h', 'Interpretation', 'Delta (pp)']].to_string(index=False))
effect_df.to_csv(OUTPUT_DIR / 'tables' / 'effect_size_analysis.csv', index=False)

---
## Study 3: Country-Level Performance Analysis

We analyze accuracy variations across the 20 countries in the evaluation set to understand geographic biases in cultural knowledge retrieval.

In [ ]:
# ============================================================================
# CELL 11: PER-COUNTRY ACCURACY TABLE
# ============================================================================

country_rows = []

for country in countries:
    idx = [i for i, c in enumerate(country_list) if c == country]
    n_country = len(idx)
    
    row_data = {'Country': country, 'N': n_country}
    
    for config in ['baseline_no_rag', 'rag_basic', 'phase6_full_system']:
        preds_c = [ablation_lists[config][i] for i in idx]
        truth_c = [correct_answers[i] for i in idx]
        correct_c = sum(p == c for p, c in zip(preds_c, truth_c))
        acc_c = correct_c / n_country * 100 if n_country > 0 else 0
        row_data[f'{CONFIG_SHORT[config]} Acc'] = round(acc_c, 1)
        row_data[f'{CONFIG_SHORT[config]} Correct'] = correct_c
    
    # RAG delta
    row_data['RAG Delta (pp)'] = round(
        row_data[f'{CONFIG_SHORT["rag_basic"]} Acc'] - row_data[f'{CONFIG_SHORT["baseline_no_rag"]} Acc'], 1)
    row_data['Full Delta (pp)'] = round(
        row_data[f'{CONFIG_SHORT["phase6_full_system"]} Acc'] - row_data[f'{CONFIG_SHORT["baseline_no_rag"]} Acc'], 1)
    
    country_rows.append(row_data)

country_df = pd.DataFrame(country_rows).sort_values('Full Delta (pp)', ascending=False)

print('Per-Country Accuracy:')
display_cols = ['Country', 'N', f'{CONFIG_SHORT["baseline_no_rag"]} Acc',
                f'{CONFIG_SHORT["rag_basic"]} Acc', f'{CONFIG_SHORT["phase6_full_system"]} Acc',
                'RAG Delta (pp)', 'Full Delta (pp)']
print(country_df[display_cols].to_string(index=False))

# Summary
helped = country_df[country_df['Full Delta (pp)'] > 0]
hurt = country_df[country_df['Full Delta (pp)'] < 0]
neutral = country_df[country_df['Full Delta (pp)'] == 0]
print(f'\nFull system vs Baseline:  Helped {len(helped)} | Hurt {len(hurt)} | Neutral {len(neutral)} countries')

country_df.to_csv(OUTPUT_DIR / 'tables' / 'accuracy_by_country.csv', index=False)

In [ ]:
# ============================================================================
# CELL 12: FIGURE 7 — Per-Country Grouped Bar Chart
# ============================================================================

fig, ax = plt.subplots(figsize=(14, 7))

sorted_countries = country_df.sort_values(f'{CONFIG_SHORT["phase6_full_system"]} Acc', ascending=True)['Country'].tolist()
x = np.arange(len(sorted_countries))
width = 0.25

baseline_accs = [country_df[country_df['Country'] == c][f'{CONFIG_SHORT["baseline_no_rag"]} Acc'].values[0] for c in sorted_countries]
rag_accs = [country_df[country_df['Country'] == c][f'{CONFIG_SHORT["rag_basic"]} Acc'].values[0] for c in sorted_countries]
full_accs = [country_df[country_df['Country'] == c][f'{CONFIG_SHORT["phase6_full_system"]} Acc'].values[0] for c in sorted_countries]

ax.barh(x - width, baseline_accs, width, label='Baseline (No RAG)', color='#d62728', alpha=0.8)
ax.barh(x, rag_accs, width, label='+ Basic RAG', color='#1f77b4', alpha=0.8)
ax.barh(x + width, full_accs, width, label='Full System', color='#2ca02c', alpha=0.8)

ax.set_yticks(x)
ax.set_yticklabels(sorted_countries)
ax.set_xlabel('Accuracy (%)')
ax.set_title('Figure 7: Per-Country Accuracy — Baseline vs. RAG vs. Full System')
ax.legend(loc='lower right')
ax.axvline(x=25, color='gray', linestyle=':', alpha=0.5, label='Random')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / 'fig7_country_accuracy.pdf', bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figures' / 'fig7_country_accuracy.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# CELL 13: FIGURE 8 — Country × Config Accuracy Heatmap
# ============================================================================

# Build heatmap data: rows = countries, cols = configs
heatmap_data = []
for country in sorted(countries):
    idx = [i for i, c in enumerate(country_list) if c == country]
    n_c = len(idx)
    row = {'Country': country}
    for config in CONFIG_ORDER:
        preds_c = [ablation_lists[config][i] for i in idx]
        truth_c = [correct_answers[i] for i in idx]
        acc = sum(p == c for p, c in zip(preds_c, truth_c)) / n_c * 100 if n_c > 0 else 0
        row[CONFIG_SHORT[config]] = round(acc, 1)
    heatmap_data.append(row)

hm_df = pd.DataFrame(heatmap_data).set_index('Country')

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(hm_df, annot=True, fmt='.0f', cmap='RdYlGn', center=60,
            linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': 'Accuracy (%)'})
ax.set_title('Figure 8: Country × Configuration Accuracy Heatmap')
ax.set_ylabel('')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / 'fig8_country_heatmap.pdf', bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figures' / 'fig8_country_heatmap.png', bbox_inches='tight')
plt.show()

hm_df.to_csv(OUTPUT_DIR / 'tables' / 'country_config_heatmap.csv')

---
## Study 4: RAG Impact Analysis

We analyze the net impact of RAG by categorizing every question into one of four outcomes: both correct, RAG fixed, RAG hurt, or both wrong.

In [ ]:
# ============================================================================
# CELL 14: RAG IMPACT — Helped vs. Hurt Analysis
# ============================================================================

baseline = ablation_lists['baseline_no_rag']
rag = ablation_lists['rag_basic']
full = ablation_lists['phase6_full_system']

# Categorize each question
categories = {'both_correct': [], 'rag_fixed': [], 'rag_hurt': [], 'both_wrong': []}

for i, (b, r, f, c) in enumerate(zip(baseline, rag, full, correct_answers)):
    qid = all_ids[i]
    country = country_list[i]
    q_text = questions_map.get(qid, '')
    
    entry = {'id': qid, 'country': country, 'question': q_text,
             'correct': c, 'baseline': b, 'rag': r, 'full': f}
    
    if b == c and r == c:
        categories['both_correct'].append(entry)
    elif b != c and r == c:
        categories['rag_fixed'].append(entry)
    elif b == c and r != c:
        categories['rag_hurt'].append(entry)
    else:
        categories['both_wrong'].append(entry)

# ── Summary ────────────────────────────────────────────────────────────────
print('RAG Impact Summary (Baseline vs Basic RAG):')
print('=' * 60)
total = N
for cat, items in categories.items():
    pct = len(items) / total * 100
    print(f'  {cat:20s}: {len(items):4d} ({pct:5.1f}%)')

net = len(categories['rag_fixed']) - len(categories['rag_hurt'])
print(f'\n  Net RAG benefit: {net:+d} questions')

# ── Full System comparison ─────────────────────────────────────────────────
full_cats = {'both_correct': 0, 'full_fixed': 0, 'full_hurt': 0, 'both_wrong': 0}
for i, (b, f, c) in enumerate(zip(baseline, full, correct_answers)):
    if b == c and f == c: full_cats['both_correct'] += 1
    elif b != c and f == c: full_cats['full_fixed'] += 1
    elif b == c and f != c: full_cats['full_hurt'] += 1
    else: full_cats['both_wrong'] += 1

print(f'\nFull System vs Baseline:')
for cat, count in full_cats.items():
    print(f'  {cat:20s}: {count:4d} ({count/total*100:5.1f}%)')
print(f"  Net benefit: {full_cats['full_fixed'] - full_cats['full_hurt']:+d} questions")

# ── Figure: Stacked bar ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

labels_bar = ['RAG Basic\nvs Baseline', 'Full System\nvs Baseline']
both_c = [len(categories['both_correct']), full_cats['both_correct']]
fixed = [len(categories['rag_fixed']), full_cats['full_fixed']]
hurt = [len(categories['rag_hurt']), full_cats['full_hurt']]
both_w = [len(categories['both_wrong']), full_cats['both_wrong']]

x_pos = np.arange(len(labels_bar))
w = 0.5

p1 = ax.bar(x_pos, both_c, w, label='Both Correct', color='#2ca02c')
p2 = ax.bar(x_pos, fixed, w, bottom=both_c, label='System Fixed', color='#1f77b4')
p3 = ax.bar(x_pos, hurt, w, bottom=[a+b for a,b in zip(both_c, fixed)], label='System Hurt', color='#d62728')
p4 = ax.bar(x_pos, both_w, w, bottom=[a+b+c for a,b,c in zip(both_c, fixed, hurt)], label='Both Wrong', color='#7f7f7f')

ax.set_xticks(x_pos)
ax.set_xticklabels(labels_bar)
ax.set_ylabel('Number of Questions')
ax.set_title('Figure 9: RAG Impact — Question Outcome Categories')
ax.legend(loc='upper right')

# Add count labels
for bars in [p1, p2, p3, p4]:
    for bar in bars:
        h = bar.get_height()
        if h > 3:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_y() + h/2,
                    f'{int(h)}', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / 'fig9_rag_impact.pdf', bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figures' / 'fig9_rag_impact.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================================
# CELL 15: RAG HURT CASES — Detailed Analysis
# ============================================================================

print('CASES WHERE RAG HURT PREDICTIONS')
print('=' * 80)
print(f'Total: {len(categories["rag_hurt"])} questions\n')

# By country
hurt_by_country = Counter(e['country'] for e in categories['rag_hurt'])
country_totals = Counter(country_list)

print(f'{"Country":<10} {"Hurt":<6} {"Total":<7} {"Hurt Rate":<10}')
print('-' * 35)
for country in sorted(hurt_by_country, key=hurt_by_country.get, reverse=True):
    h = hurt_by_country[country]
    t = country_totals[country]
    print(f'{country:<10} {h:<6} {t:<7} {h/t*100:>5.1f}%')

# Detailed cases
print(f'\nDetailed RAG Hurt Cases (showing all {len(categories["rag_hurt"])}):') 
print('-' * 80)
for i, case in enumerate(categories['rag_hurt'], 1):
    opts = options_map.get(case['id'], {})
    print(f"\n{i}. [{case['id']}] ({case['country']})")
    print(f"   Q: {case['question'][:90]}")
    print(f"   Correct: {case['correct']} ({opts.get(case['correct'], '?')})")
    print(f"   Baseline: {case['baseline']} ✓  |  RAG: {case['rag']} ✗  |  Full: {case['full']} {'✓' if case['full'] == case['correct'] else '✗'}")

# Save
pd.DataFrame(categories['rag_hurt']).to_csv(OUTPUT_DIR / 'tables' / 'rag_hurt_cases.csv', index=False)
pd.DataFrame(categories['rag_fixed']).to_csv(OUTPUT_DIR / 'tables' / 'rag_fixed_cases.csv', index=False)
print(f'\nSaved rag_hurt_cases.csv and rag_fixed_cases.csv')

---
## Study 5: Error Analysis & Taxonomy

We categorize all errors from the best-performing configuration using an automated error taxonomy:
- **Retrieval Failure** — No relevant context retrieved
- **Reasoning Failure** — Context is present but model misinterprets
- **Knowledge Gap** — Information missing from KB
- **Ambiguous** — Question allows multiple valid interpretations

In [ ]:
# ============================================================================
# CELL 16: ERROR TAXONOMY — Automated Categorization
# ============================================================================

# Collect errors from the full system
errors_full = []
for i in range(N):
    pred = ablation_lists['phase6_full_system'][i]
    corr = correct_answers[i]
    if pred != corr:
        qid = all_ids[i]
        question = questions_map.get(qid, '')
        q_lower = question.lower()
        country = country_list[i]
        
        # Auto-categorize using heuristics
        if any(neg in q_lower for neg in ['not', 'except', 'never', 'which is not', 'least']):
            category = 'reasoning_failure'
            evidence = 'Negation/exception in question'
        elif any(t in q_lower for t in ['current', 'now', 'recent', 'latest', 'today', '2024', '2025']):
            category = 'knowledge_gap'
            evidence = 'Temporal/current info requested'
        elif ablation_lists['baseline_no_rag'][i] == corr and pred != corr:
            category = 'reasoning_failure'
            evidence = 'Baseline correct but full system wrong (RAG confused model)'
        elif ablation_lists['rag_basic'][i] == corr and pred != corr:
            category = 'reasoning_failure'
            evidence = 'Basic RAG correct, advanced features caused regression'
        elif ablation_lists['baseline_no_rag'][i] != corr and ablation_lists['rag_basic'][i] != corr:
            category = 'knowledge_gap'
            evidence = 'Neither baseline nor RAG could answer'
        else:
            category = 'ambiguous'
            evidence = 'Unclear failure pattern'
        
        errors_full.append({
            'id': qid,
            'country': country,
            'question': question,
            'predicted': pred,
            'correct': corr,
            'category': category,
            'evidence': evidence,
            'baseline_pred': ablation_lists['baseline_no_rag'][i],
            'rag_pred': ablation_lists['rag_basic'][i],
        })

error_df = pd.DataFrame(errors_full)
total_errors = len(error_df)

print(f'Total errors (Full System): {total_errors} / {N} ({total_errors/N*100:.1f}%)')
print(f'\nError Category Breakdown:')
print('=' * 60)

cat_counts = error_df['category'].value_counts()
for cat, count in cat_counts.items():
    pct_err = count / total_errors * 100
    pct_all = count / N * 100
    bar = '█' * int(pct_err / 2)
    print(f'  {cat:25s} {count:4d}  ({pct_err:5.1f}% of errors, {pct_all:5.1f}% of total)  {bar}')

# ── Figure: Error Distribution ─────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
colors_pie = {'reasoning_failure': '#ff7f0e', 'knowledge_gap': '#1f77b4',
              'ambiguous': '#7f7f7f', 'retrieval_failure': '#d62728', 'data_quality': '#9467bd'}
cats = cat_counts.index.tolist()
vals = cat_counts.values.tolist()
pie_colors = [colors_pie.get(c, '#aaaaaa') for c in cats]

wedges, texts, autotexts = ax1.pie(vals, labels=cats, autopct='%1.1f%%',
                                    colors=pie_colors, startangle=90)
for t in autotexts:
    t.set_fontsize(9)
ax1.set_title('Error Category Distribution')

# Bar chart by country
err_by_country = error_df.groupby('country').size().sort_values(ascending=True)
total_by_country = pd.Series(Counter(country_list))
err_rate = (err_by_country / total_by_country * 100).sort_values(ascending=True)

ax2.barh(err_rate.index, err_rate.values, color='#d62728', alpha=0.7)
ax2.set_xlabel('Error Rate (%)')
ax2.set_title('Error Rate by Country')
ax2.axvline(x=total_errors/N*100, color='black', linestyle='--', alpha=0.5, label='Overall avg')
ax2.legend(fontsize=8)

fig.suptitle('Figure 10: Error Analysis', fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / 'fig10_error_analysis.pdf', bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figures' / 'fig10_error_analysis.png', bbox_inches='tight')
plt.show()

error_df.to_csv(OUTPUT_DIR / 'tables' / 'error_cases_full_system.csv', index=False)

In [ ]:
# ============================================================================
# CELL 17: ERROR EXAMPLES — Per Category
# ============================================================================

for cat in cat_counts.index:
    cat_errors = error_df[error_df['category'] == cat]
    print(f'\n{"=" * 70}')
    print(f'{cat.upper()} — {len(cat_errors)} cases')
    print(f'{"=" * 70}')
    
    for _, err in cat_errors.head(3).iterrows():
        opts = options_map.get(err['id'], {})
        print(f"\n  [{err['id']}] ({err['country']})")
        print(f"  Q: {err['question'][:85]}")
        print(f"  Correct: {err['correct']} ({opts.get(err['correct'], '?')})")
        print(f"  Predicted: {err['predicted']} ({opts.get(err['predicted'], '?')})")
        print(f"  Baseline: {err['baseline_pred']}  |  RAG: {err['rag_pred']}")
        print(f"  Evidence: {err['evidence']}")

In [ ]:
# ============================================================================
# CELL 18: ERROR × COUNTRY CROSSTAB
# ============================================================================

if len(error_df) > 0:
    crosstab = pd.crosstab(error_df['country'], error_df['category'])
    
    fig, ax = plt.subplots(figsize=(12, 8))
    crosstab_pct = crosstab.div(crosstab.sum(axis=1), axis=0) * 100
    crosstab_pct.plot(kind='barh', stacked=True, ax=ax, 
                      color=[colors_pie.get(c, '#aaa') for c in crosstab.columns])
    ax.set_xlabel('Percentage of Errors (%)')
    ax.set_title('Figure 11: Error Category Distribution by Country')
    ax.legend(title='Error Category', bbox_to_anchor=(1.02, 1), loc='upper left')
    
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / 'figures' / 'fig11_error_by_country.pdf', bbox_inches='tight')
    fig.savefig(OUTPUT_DIR / 'figures' / 'fig11_error_by_country.png', bbox_inches='tight')
    plt.show()
    
    crosstab.to_csv(OUTPUT_DIR / 'tables' / 'error_country_crosstab.csv')
else:
    print('No errors to analyze.')

---
## Study 6: Prediction Agreement & Stability

We measure how often different configurations agree on the same prediction to quantify system stability across ablation phases.

In [ ]:
# ============================================================================
# CELL 19: FIGURE 12 — Cross-Config Agreement Matrix
# ============================================================================

n_configs = len(CONFIG_ORDER)
agree_matrix = np.zeros((n_configs, n_configs))

for i in range(n_configs):
    for j in range(n_configs):
        cA, cB = CONFIG_ORDER[i], CONFIG_ORDER[j]
        agree = sum(a == b for a, b in zip(ablation_lists[cA], ablation_lists[cB]))
        agree_matrix[i, j] = agree / N * 100

fig, ax = plt.subplots(figsize=(10, 8))
short_labels = [CONFIG_SHORT[c] for c in CONFIG_ORDER]

sns.heatmap(agree_matrix, xticklabels=short_labels, yticklabels=short_labels,
            annot=True, fmt='.1f', cmap='YlGnBu', vmin=60, vmax=100,
            linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': 'Agreement (%)'})
ax.set_title('Figure 12: Pairwise Prediction Agreement Matrix (%)')

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / 'fig12_agreement_matrix.pdf', bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figures' / 'fig12_agreement_matrix.png', bbox_inches='tight')
plt.show()

# Summary Stats
print('Agreement Statistics:')
for i in range(n_configs - 1):
    j = i + 1
    print(f'  {CONFIG_SHORT[CONFIG_ORDER[i]]:>12} ↔ {CONFIG_SHORT[CONFIG_ORDER[j]]:<12}: {agree_matrix[i,j]:.1f}%')

# Baseline vs Full
print(f'\n  Baseline ↔ Full System: {agree_matrix[0, -1]:.1f}%')

# Questions where ALL configs agree
all_agree = sum(1 for i in range(N)
                if len(set(ablation_lists[c][i] for c in CONFIG_ORDER)) == 1)
print(f'  All configs agree: {all_agree}/{N} ({all_agree/N*100:.1f}%)')

In [ ]:
# ============================================================================
# CELL 20: PREDICTION DISTRIBUTION ANALYSIS
# ============================================================================

# Answer distribution per config
dist_data = {'Ground Truth': Counter(correct_answers)}
for config in CONFIG_ORDER:
    dist_data[CONFIG_SHORT[config]] = Counter(ablation_lists[config])

dist_df = pd.DataFrame(dist_data).fillna(0).astype(int)
dist_df = dist_df.reindex(['A', 'B', 'C', 'D'])

print('Answer Distribution per Configuration:')
print(dist_df.to_string())

# ── Figure: Distribution comparison ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(4)  # A, B, C, D
width = 0.08
configs_to_plot = ['Ground Truth'] + [CONFIG_SHORT[c] for c in CONFIG_ORDER]

for i, col in enumerate(configs_to_plot):
    vals = [dist_df.loc[letter, col] for letter in ['A', 'B', 'C', 'D']]
    offset = (i - len(configs_to_plot)/2) * width
    ax.bar(x + offset, vals, width, label=col, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(['A', 'B', 'C', 'D'])
ax.set_ylabel('Count')
ax.set_title('Figure 13: Answer Distribution — Ground Truth vs. Predictions')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / 'fig13_answer_distribution.pdf', bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figures' / 'fig13_answer_distribution.png', bbox_inches='tight')
plt.show()

---
## Study 7: Inference Timing Analysis

In [ ]:
# ============================================================================
# CELL 21: THROUGHPUT & TIMING ANALYSIS
# ============================================================================

timing_rows = []
for config in CONFIG_ORDER:
    t = timing_map.get(config, 0)
    timing_rows.append({
        'Config': CONFIG_SHORT[config],
        'Time (s)': round(t, 1),
        'Time/Q (ms)': round(t / N * 1000, 1) if N > 0 else 0,
        'Q/sec': round(N / t, 1) if t > 0 else 0,
    })

timing_table = pd.DataFrame(timing_rows)
print('Inference Timing:')
print(timing_table.to_string(index=False))

# ── Figure ─────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

labels = timing_table['Config']
times = timing_table['Time (s)']
qps = timing_table['Q/sec']

ax1.barh(labels, times, color=sns.color_palette('coolwarm', len(labels)))
ax1.set_xlabel('Total Time (seconds)')
ax1.set_title('Inference Time per Config')

ax2.barh(labels, qps, color=sns.color_palette('coolwarm_r', len(labels)))
ax2.set_xlabel('Questions per Second')
ax2.set_title('Inference Throughput')

fig.suptitle('Figure 14: Inference Timing Analysis', fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / 'fig14_timing.pdf', bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figures' / 'fig14_timing.png', bbox_inches='tight')
plt.show()

timing_table.to_csv(OUTPUT_DIR / 'tables' / 'inference_timing.csv', index=False)

---
## Study 8: Confusion Matrix & Answer Patterns

In [ ]:
# ============================================================================
# CELL 22: CONFUSION MATRICES — Baseline vs Full System
# ============================================================================

from sklearn.metrics import confusion_matrix as sk_cm

labels_cm = ['A', 'B', 'C', 'D']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Baseline confusion matrix
cm_base = sk_cm(correct_answers, ablation_lists['baseline_no_rag'], labels=labels_cm)
sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=labels_cm, yticklabels=labels_cm,
            linewidths=0.5, linecolor='white')
ax1.set_xlabel('Predicted')
ax1.set_ylabel('True')
ax1.set_title('Baseline (No RAG)')

# Full system confusion matrix
cm_full = sk_cm(correct_answers, ablation_lists['phase6_full_system'], labels=labels_cm)
sns.heatmap(cm_full, annot=True, fmt='d', cmap='Greens', ax=ax2,
            xticklabels=labels_cm, yticklabels=labels_cm,
            linewidths=0.5, linecolor='white')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('True')
ax2.set_title('Full System')

fig.suptitle('Figure 15: Confusion Matrices', fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figures' / 'fig15_confusion_matrices.pdf', bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figures' / 'fig15_confusion_matrices.png', bbox_inches='tight')
plt.show()

# Per-class accuracy
print('Per-Answer-Class Accuracy:')
print(f'{"Answer":<8} {"Baseline":>10} {"Full System":>12}')
print('-' * 35)
for k, label in enumerate(labels_cm):
    base_acc = cm_base[k, k] / cm_base[k].sum() * 100 if cm_base[k].sum() > 0 else 0
    full_acc = cm_full[k, k] / cm_full[k].sum() * 100 if cm_full[k].sum() > 0 else 0
    print(f'{label:<8} {base_acc:>9.1f}% {full_acc:>11.1f}%')

---
## Export: LaTeX Tables & Final Summary

In [ ]:
# ============================================================================
# CELL 23: GENERATE LaTeX TABLES FOR PAPER
# ============================================================================

# ── Table 1: Ablation Results ─────────────────────────────────────────────
latex1 = ablation_df[['Label', 'Accuracy (%)', 'Delta (pp)', 'CI Lower', 'CI Upper']].copy()
latex1['95% CI'] = latex1.apply(lambda r: f"[{r['CI Lower']:.1f}, {r['CI Upper']:.1f}]", axis=1)
latex1 = latex1[['Label', 'Accuracy (%)', 'Delta (pp)', '95% CI']]
latex1.columns = ['Configuration', 'Accuracy (\\%)', '$\\Delta$ (pp)', '95\\% CI']

latex_str1 = latex1.to_latex(index=False, escape=False, column_format='lrrr',
                              caption='Ablation study results showing incremental accuracy gains.',
                              label='tab:ablation')

with open(OUTPUT_DIR / 'tables' / 'table1_ablation.tex', 'w') as f:
    f.write(latex_str1)
print('Table 1 (Ablation):')
print(latex_str1)

# ── Table 2: Statistical Significance (Consecutive) ───────────────────────
consec_pairs = []
for i in range(len(CONFIG_ORDER) - 1):
    cA, cB = CONFIG_ORDER[i], CONFIG_ORDER[i+1]
    mask = (pairwise_df['Config A'] == CONFIG_SHORT[cA]) & (pairwise_df['Config B'] == CONFIG_SHORT[cB])
    if mask.any():
        r = pairwise_df[mask].iloc[0]
        consec_pairs.append({
            'Transition': f"{CONFIG_SHORT[cA]} $\\to$ {CONFIG_SHORT[cB]}",
            '$\\Delta$ (pp)': f"{r['Delta (pp)']:+.2f}",
            'p-value': f"{r['p-value']:.4f}",
            'Sig.': r['Significant'],
        })

if consec_pairs:
    latex2_df = pd.DataFrame(consec_pairs)
    latex_str2 = latex2_df.to_latex(index=False, escape=False, column_format='lrrl',
                                     caption="McNemar's test for consecutive ablation phases.",
                                     label='tab:significance')
    with open(OUTPUT_DIR / 'tables' / 'table2_significance.tex', 'w') as f:
        f.write(latex_str2)
    print('\nTable 2 (Significance):')
    print(latex_str2)

# ── Table 3: Effect Size ──────────────────────────────────────────────────
consec_effects = effect_df[effect_df['Type'] == 'Consecutive'][['Comparison', 'h', 'Interpretation', 'Delta (pp)']].copy()
consec_effects.columns = ['Transition', "Cohen's $h$", 'Effect Size', '$\\Delta$ (pp)']
latex_str3 = consec_effects.to_latex(index=False, escape=False, column_format='lrll',
                                      caption="Cohen's $h$ effect sizes for consecutive phase transitions.",
                                      label='tab:effectsize')
with open(OUTPUT_DIR / 'tables' / 'table3_effect_size.tex', 'w') as f:
    f.write(latex_str3)
print('\nTable 3 (Effect Size):')
print(latex_str3)

print(f'\nAll LaTeX tables saved to {OUTPUT_DIR}/tables/')

In [ ]:
# ============================================================================
# CELL 24: FINAL SUMMARY
# ============================================================================

print('=' * 70)
print('RESEARCH STUDIES — FINAL SUMMARY')
print('=' * 70)

print(f'\nDataset: {N} questions, {len(countries)} countries')
print(f'Ablation configs: {len(CONFIG_ORDER)}')
print(f'Baseline accuracy: {baseline_acc:.2f}%')
print(f'Best accuracy:     {best_acc:.2f}%  (Full System)')
print(f'Total gain:        +{total_gain:.2f} pp')

# Key findings
print(f'\nKey Findings:')
gains_sorted = ablation_df[ablation_df['Delta (pp)'] > 0].sort_values('Delta (pp)', ascending=False)
if len(gains_sorted) > 0:
    top = gains_sorted.iloc[0]
    print(f'  Largest single gain: {top["Label"]} (+{top["Delta (pp)"]:.2f} pp)')

# Baseline vs Full significance
base_full_mask = (pairwise_df['Config A'] == CONFIG_SHORT['baseline_no_rag']) & \
                 (pairwise_df['Config B'] == CONFIG_SHORT['phase6_full_system'])
if base_full_mask.any():
    bf = pairwise_df[base_full_mask].iloc[0]
    print(f"  Baseline→Full significance: p={bf['p-value']:.4f} ({bf['Significant']})")

print(f'\n  RAG net benefit: {len(categories["rag_fixed"])} fixed - {len(categories["rag_hurt"])} hurt = {len(categories["rag_fixed"]) - len(categories["rag_hurt"]):+d}')
print(f'  Error categories: {dict(cat_counts)}')

# Output files
print(f'\nOutput Files:')
for subdir in ['figures', 'tables']:
    path = OUTPUT_DIR / subdir
    files = sorted(path.glob('*'))
    print(f'  {subdir}/ ({len(files)} files):')
    for f in files:
        size_kb = f.stat().st_size / 1024
        print(f'    {f.name} ({size_kb:.1f} KB)')

print(f'\n{"=" * 70}')
print('All studies complete. Figures and tables ready for paper.')
print(f'{"=" * 70}')